# MNIST Classifier

Training a LeNet-5 convolutional network on MNIST with PyTorch Lightning.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = ['retina', 'svg']

import os
from collections import Counter

import matplotlib.pyplot as plt
import torch
from lightning import Trainer
from torchinfo import summary
from tqdm.auto import tqdm

from chimera.data import MNISTDataModule
from chimera.modules import ClassifierModule
from chimera.optim import LinearWarmupCosineAnnealingLR

os.environ["DATA_DIR"] = "../../../data"

## Data

Load MNIST and inspect the label distribution across the training and validation splits.

In [ ]:
dm = MNISTDataModule(pin_memory=False, num_workers=0, data_dir=os.environ["DATA_DIR"])
dm.prepare_data()
dm.setup("fit")

train_loader = dm.train_dataloader()
val_loader = dm.val_dataloader()

In [ ]:
train_label_counts = Counter()
for _, labels in tqdm(train_loader, desc="Counting training labels"):
    train_label_counts.update(labels.tolist())

val_label_counts = Counter()
for _, labels in tqdm(val_loader, desc="Counting validation labels"):
    val_label_counts.update(labels.tolist())

plt.figure(figsize=(12, 5))
plt.suptitle("MNIST Label Distribution")

plt.subplot(1, 2, 1)
plt.bar(list(train_label_counts.keys()), list(train_label_counts.values()))
plt.title("Training Label Distribution")
plt.xlabel("Label")
plt.ylabel("Count")

plt.subplot(1, 2, 2)
plt.bar(list(val_label_counts.keys()), list(val_label_counts.values()))
plt.title("Validation Label Distribution")
plt.xlabel("Label")
plt.ylabel("Count")

plt.show()

## Model

LeNet-5 with tanh activations and average pooling. Inputs are standardized in the forward pass.

In [ ]:
from chimera.models import LeNet5

model = LeNet5(in_channels=1, num_classes=10)
summary(
    model,
    input_size=(1, 1, 28, 28),
    col_names=["output_size", "mult_adds", "num_params"],
    verbose=0,
)

## Training

Wrap the model in a `LightningModule` and train with a linear-warmup cosine-annealing schedule.

In [ ]:
N_EPOCH = 10

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = LinearWarmupCosineAnnealingLR(
    optimizer,
    warmup_steps=1000,
    n_epochs=N_EPOCH,
    train_loader=train_loader,
)

classifier_module = ClassifierModule(model, optimizer, scheduler)

trainer = Trainer(
    max_epochs=N_EPOCH,
    precision="bf16-true",
)

trainer.fit(
    classifier_module, train_dataloaders=train_loader, val_dataloaders=val_loader
)
trainer.test(classifier_module, dataloaders=val_loader)

## Analysis

Inspect the training run: plot the logged loss/accuracy curves, then evaluate the
trained model on the validation set to build a confusion matrix.

In [ ]:
import numpy as np

metrics = np.genfromtxt(
    f"{trainer.logger.log_dir}/metrics.csv", delimiter=",", names=True
)


def series(step_key, value_key):
    step = metrics[step_key]
    value = metrics[value_key]
    mask = ~np.isnan(value)
    return step[mask], value[mask]


fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Training Metrics")

for ax, metric, title in zip(axes, ["loss", "acc"], ["Loss", "Accuracy"]):
    train_step, train_val = series("step", f"train{metric}_epoch")
    val_step, val_val = series("step", f"val{metric}")
    ax.plot(train_step, train_val, marker="o", label="train")
    ax.plot(val_step, val_val, marker="o", label="val")
    ax.set_title(title)
    ax.set_xlabel("Step")
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(alpha=0.3)

plt.show()

In [ ]:
num_classes = 10

classifier_module.eval()
device = classifier_module.device

confusion = torch.zeros(num_classes, num_classes, dtype=torch.long)
with torch.no_grad():
    for x, y in tqdm(val_loader, desc="Evaluating"):
        preds = (
            classifier_module(x.to(device, dtype=classifier_module.dtype))
            .argmax(dim=1)
            .cpu()
        )
        for t, p in zip(y, preds):
            confusion[t, p] += 1

accuracy = confusion.diag().sum().item() / confusion.sum().item()
print(f"Validation accuracy: {accuracy:.4f}")

confusion_np = confusion.numpy()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(confusion_np, cmap="Blues")
fig.colorbar(im, ax=ax)

ax.set_title("Validation Confusion Matrix")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_xticks(range(num_classes))
ax.set_yticks(range(num_classes))

threshold = confusion_np.max() / 2
for i in range(num_classes):
    for j in range(num_classes):
        ax.text(
            j,
            i,
            confusion_np[i, j],
            ha="center",
            va="center",
            color="white" if confusion_np[i, j] > threshold else "black",
        )

plt.tight_layout()
plt.show()